In [1]:
#Cargamos los CSV de las 7 estaciones en una sola tabla y contamos los dias con dato de lluvia por estacion y año, para ver donde están
#los huecos.

import pandas as pd
from pathlib import Path

clima = pd.concat(
    [pd.read_csv(f, dtype=str) for f in Path("../data/raw/aemet").glob("*.csv")],
    ignore_index=True,
)
clima["fecha"] = pd.to_datetime(clima["fecha"])

print(clima.shape)
print(clima.columns.tolist())

cobertura = clima.dropna(subset=["prec"]).groupby([clima["fecha"].dt.year, "indicativo"]).size().unstack()
cobertura

(79277, 27)
['fecha', 'indicativo', 'nombre', 'provincia', 'altitud', 'tmed', 'prec', 'tmin', 'horatmin', 'tmax', 'horatmax', 'dir', 'velmedia', 'racha', 'horaracha', 'sol', 'presMax', 'horaPresMax', 'presMin', 'horaPresMin', 'hrMedia', 'pintMax', 'horaPIntMax', 'hrMax', 'horaHrMax', 'hrMin', 'horaHrMin']


indicativo,5192,5270B,5402,5514,5530E,5783,5796
fecha,,,,,,,
1993,184.0,184.0,184.0,184.0,184.0,184.0,184.0
1994,365.0,365.0,365.0,365.0,365.0,365.0,365.0
1995,30.0,365.0,365.0,365.0,365.0,365.0,365.0
1996,NaN,366.0,366.0,366.0,366.0,366.0,366.0
1997,61.0,365.0,365.0,365.0,365.0,365.0,365.0
1998,274.0,365.0,365.0,365.0,365.0,365.0,365.0
1999,274.0,365.0,365.0,365.0,365.0,365.0,365.0
2000,305.0,366.0,366.0,366.0,366.0,366.0,366.0
2001,31.0,365.0,365.0,365.0,365.0,365.0,365.0


In [2]:
#Nos quedamos con las 5 estaciones elegidas y convertimos la lluvia y temperatura a número ("ip" = 0 y coma decimal) para poder calcular
#con ella

ESTACIONES = ["5270B", "5402", "5530E", "5783", "5796"]
clima = clima[clima["indicativo"].isin(ESTACIONES)].copy()

clima["prec"] = clima["prec"].replace("Ip", "0")
for col in ["prec", "tmed"]:
    clima[col] = pd.to_numeric(clima[col].str.replace(",", "."), errors="coerce")

clima = clima[["fecha", "indicativo", "nombre", "prec", "tmed"]]
print(clima.isna().sum())
clima.describe()

fecha           0
indicativo      0
nombre          0
prec            7
tmed          217
dtype: int64


,fecha,prec,tmed
count,60825,60818.000000,60608.000000
mean,2010-02-24 00:00:00,1.335379,18.221194
min,1993-07-01 00:00:00,0.000000,-3.100000
25%,2001-10-28 00:00:00,0.000000,12.300000
50%,2010-02-24 00:00:00,0.000000,17.700000
75%,2018-06-23 00:00:00,0.000000,24.300000
max,2026-10-20 00:00:00,154.300000,36.800000
std,NaN,5.030523,7.187725


In [3]:
#Hacemos la media diaria de las 5 estaciones y la pasamos a semanas que terminan en martes (lluvia sumada, temperatura media), para 
#tener el mismo calendario que los embalses.

diario = clima.groupby("fecha")[["prec", "tmed"]].mean()

semanal = diario.resample("W-TUE").agg(
    lluvia=("prec", "sum"),
    tmed=("tmed", "mean"),
    dias=("prec", "count"),
)

print(semanal["dias"].value_counts())
semanal.head()

dias
7    1737
6       1
Name: count, dtype: int64


,lluvia,tmed,dias
fecha,,,
1993-07-06,0.0,25.573333,6
1993-07-13,0.0,26.337143,7
1993-07-20,0.0,28.428571,7
1993-07-27,0.0,29.046429,7
1993-08-03,0.0,29.110714,7


In [4]:
#Guardamos la serie semanal de clima y comprobamos que sus fechas coinciden con las de la cuenca, para poder curzarlas sin perder semanas.

semanal = semanal.drop(columns="dias")
semanal.to_parquet("../data/processed/clima_semanal.parquet")

cuenca = pd.read_parquet("../data/processed/cuenca_semanal.parquet")
comunes = cuenca.index.intersection(semanal.index)
print("Semanas cuenca:", len(cuenca), "| coinciden con clima:", len(comunes))

Semanas cuenca: 1708 | coinciden con clima: 1708
